# DeepCas13: exclude `N`, exact preprocessing, and fixed split

This notebook uses the authors' released `data/training_data.csv` and removes
every sample whose 33-nt guide sequence contains `N` **before** label
transformation and before the train/validation/unseen split.

For the retained samples, it reproduces:

- the authors' raw LFC input;
- the exact LFC-to-`y_value` logistic transformation;
- the requested seed-42 split.

The split sizes are calculated dynamically after filtering:

- 80% of retained samples are selected as the seen set;
- 20% of the seen set becomes validation;
- all remaining samples form the unseen set.


In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_FILE = Path("data/training_data.csv")

SPLIT_SEED = 42
SEEN_FRACTION = 0.80
VALIDATION_FRACTION_WITHIN_SEEN = 0.20

OUTPUT_DIR = Path(
    "results/deepcas13_exact_no_N_fixed_split"
)
FULL_DIR = OUTPUT_DIR / "full_dataset"
SPLIT_DIR = OUTPUT_DIR / "saved_splits"

FULL_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Missing {DATA_FILE.resolve()}. "
        "Run this notebook from the repository root."
    )


## Read the authors' two-column training file

In [ ]:
data = pd.read_csv(
    DATA_FILE,
    names=["seq", "LFC"],
    header=None,
    index_col=None,
)

data["seq"] = (
    data["seq"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace("U", "T", regex=False)
)

data["LFC"] = pd.to_numeric(
    data["LFC"],
    errors="coerce",
)

valid = (
    data["seq"]
    .str.fullmatch(r"[ACGTN]+")
    .fillna(False)
    & data["LFC"].notna()
)

if not valid.all():
    raise ValueError(
        f"{int((~valid).sum())} invalid rows were found."
    )

if not data["seq"].str.len().eq(33).all():
    raise ValueError(
        "The released DeepCas13 training sequences "
        "are expected to be exactly 33 nt."
    )

original_sample_count = len(data)

contains_N = data["seq"].str.contains(
    "N",
    regex=False,
)

excluded_N_count = int(
    contains_N.sum()
)

data = (
    data.loc[~contains_N]
    .copy()
    .reset_index(drop=True)
)

if data["seq"].str.contains(
    "N",
    regex=False,
).any():
    raise RuntimeError(
        "At least one retained guide still contains N."
    )

if len(data) < 5:
    raise ValueError(
        "Fewer than five samples remain after excluding guides containing N."
    )

print("Original rows:", original_sample_count)
print("Rows excluded for containing N:", excluded_N_count)
print("Rows retained:", len(data))
print(
    "Sequence lengths:",
    data["seq"].str.len().value_counts().to_dict(),
)
print(
    "LFC range:",
    float(data["LFC"].min()),
    "to",
    float(data["LFC"].max()),
)
display(data.head())


## Apply the authors' exact LFC-to-`y_value` transformation


In [ ]:
x1 = -0.3
y1 = 0.7
x2 = 0.0
y2 = 0.3
param_n = 1.0

param_a = (
    np.log(1.0 / (1.0 - y2) - 1.0)
    - np.log(1.0 / (1.0 - y1) - 1.0)
) / (
    param_n * x1
    - param_n * x2
)

param_b = (
    -1.0
    * np.log(
        1.0 / (1.0 - y1) - 1.0
    )
    / param_a
    - param_n * x1
)

data["y_value"] = [
    1.0
    - 1.0
    / (
        1.0
        + np.exp(
            -1.0
            * param_a
            * (
                param_n * value
                + param_b
            )
        )
    )
    for value in data["LFC"].to_list()
]

data["sample_id"] = np.arange(
    len(data),
    dtype=int,
)

print("param_a:", param_a)
print("param_b:", param_b)
print(
    "y_value range:",
    float(data["y_value"].min()),
    "to",
    float(data["y_value"].max()),
)
display(data.head())


## Apply the fixed split

In [ ]:
TOTAL_SAMPLES = len(data)
SEEN_SIZE = int(
    SEEN_FRACTION * TOTAL_SAMPLES
)

np.random.seed(SPLIT_SEED)

full_indices = np.arange(
    TOTAL_SAMPLES
)

selected_seen_indices = np.random.choice(
    len(full_indices),
    size=SEEN_SIZE,
    replace=False,
)

unseen_indices = np.setdiff1d(
    full_indices,
    selected_seen_indices,
)

train_indices, validation_indices = train_test_split(
    selected_seen_indices,
    test_size=VALIDATION_FRACTION_WITHIN_SEEN,
    random_state=SPLIT_SEED,
)

train_data = data.iloc[
    train_indices
].reset_index(drop=True)

validation_data = data.iloc[
    validation_indices
].reset_index(drop=True)

unseen_data = data.iloc[
    unseen_indices
].reset_index(drop=True)

assert (
    len(train_data)
    + len(validation_data)
    == SEEN_SIZE
)

assert (
    len(train_data)
    + len(validation_data)
    + len(unseen_data)
    == TOTAL_SAMPLES
)

assert set(train_indices).isdisjoint(
    validation_indices
)
assert set(train_indices).isdisjoint(
    unseen_indices
)
assert set(validation_indices).isdisjoint(
    unseen_indices
)

for subset_name, subset_frame in [
    ("train", train_data),
    ("validation", validation_data),
    ("unseen", unseen_data),
]:
    if subset_frame["seq"].str.contains(
        "N",
        regex=False,
    ).any():
        raise RuntimeError(
            f"The {subset_name} subset contains a guide with N."
        )

display(
    pd.DataFrame(
        {
            "subset": [
                "train",
                "validation",
                "unseen",
            ],
            "n_samples": [
                len(train_data),
                len(validation_data),
                len(unseen_data),
            ],
            "fraction_of_filtered_data": [
                len(train_data) / TOTAL_SAMPLES,
                len(validation_data) / TOTAL_SAMPLES,
                len(unseen_data) / TOTAL_SAMPLES,
            ],
        }
    )
)


## Save full data, subsets, indices, and manifest

In [ ]:
data.to_csv(
    FULL_DIR / "complete_training_dataset.csv",
    index=False,
)

data["seq"].to_csv(
    FULL_DIR / "guide_sequences.txt",
    index=False,
    header=False,
)

data["LFC"].to_csv(
    FULL_DIR / "raw_LFC.txt",
    index=False,
    header=False,
)

data["y_value"].to_csv(
    FULL_DIR / "transformed_y_value.txt",
    index=False,
    header=False,
)

train_file = SPLIT_DIR / "train_split.csv"
validation_file = SPLIT_DIR / "validation_split.csv"
unseen_file = SPLIT_DIR / "unseen_split.csv"

train_data.to_csv(train_file, index=False)
validation_data.to_csv(validation_file, index=False)
unseen_data.to_csv(unseen_file, index=False)

np.savetxt(
    OUTPUT_DIR / "selected_seen_indices.txt",
    selected_seen_indices,
    fmt="%d",
)

np.savetxt(
    OUTPUT_DIR / "train_indices.txt",
    train_indices,
    fmt="%d",
)

np.savetxt(
    OUTPUT_DIR / "validation_indices.txt",
    validation_indices,
    fmt="%d",
)

np.savetxt(
    OUTPUT_DIR / "unseen_indices.txt",
    unseen_indices,
    fmt="%d",
)


def sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


manifest = {
    "source_file": str(DATA_FILE),
    "source_sha256": sha256(DATA_FILE),
    "original_samples": int(original_sample_count),
    "excluded_samples_containing_N": int(excluded_N_count),
    "total_samples_after_N_filter": int(TOTAL_SAMPLES),
    "N_filter_applied_before_split": True,
    "sequence_length": 33,
    "label_raw": "LFC",
    "label_training": "y_value",
    "label_transform": {
        "x1": x1,
        "y1": y1,
        "x2": x2,
        "y2": y2,
        "param_n": param_n,
        "param_a": float(param_a),
        "param_b": float(param_b),
    },
    "split_seed": SPLIT_SEED,
    "n_train": int(len(train_data)),
    "n_validation": int(len(validation_data)),
    "n_unseen": int(len(unseen_data)),
    "train_sha256": sha256(train_file),
    "validation_sha256": sha256(validation_file),
    "unseen_sha256": sha256(unseen_file),
}

with open(
    OUTPUT_DIR / "split_manifest.json",
    "w",
) as handle:
    json.dump(manifest, handle, indent=2)

print("Saved to:", OUTPUT_DIR.resolve())
